# Sketch Matcher — Siamese CNN for Sketch-to-Photo Matching

**What:** Draw a sketch → Pi 5 camera captures it → matches to closest photo from 10,000+ database

**No Google Drive needed.** Everything stays in Colab VM. Download results at end.

In [ ]:
# ============================================================
# STEP 0: Setup — Install dependencies
# ============================================================
import os, sys, zipfile, shutil
from pathlib import Path
from google.colab import files

!pip install -q tensorflow opencv-python numpy scikit-learn tqdm kagglehub

# Create project directory
!rm -rf /content/sketch_matcher
!mkdir -p /content/sketch_matcher
os.chdir('/content/sketch_matcher')
print(f"Working: {os.getcwd()}")

# Check GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("WARNING: No GPU! Training will be slow.")

---
## Step 1: Upload source code

Upload the `src/` folder zip from your computer.

**How to create the zip:**
1. Go to your project folder on your computer
2. Zip ONLY the `src/` folder
3. Upload it below

In [ ]:
# ============================================================
# STEP 1: Upload src.zip
# ============================================================
print("Please upload src.zip from your computer...")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zf:
            zf.extractall('/content/sketch_matcher/')
        print(f"Extracted: {filename}")

# Verify
!ls -la src/

In [ ]:
# ============================================================
# Verify all source files are present
# ============================================================
expected = ['config.py', 'class_mapping.py', 'download_data.py', 'preprocess.py',
            'data_loader.py', 'model.py', 'train.py', 'evaluate.py', 'export_tflite.py']
for f in expected:
    path = Path('/content/sketch_matcher/src') / f
    print(f"  {'OK' if path.exists() else 'MISSING'} {f}")

---
## Step 2: Download datasets
Downloads Sketchy Dataset (canonical) + TU-Berlin + QuickDraw + ImageNet-Sketch (extra sketch variety).

**If auto-download fails:** The cell below will prompt you to upload `sketchy_dataset.zip`
downloaded manually from https://www.kaggle.com/datasets/balraj98/sketchydataset

**Note:** The merged dataset holds ~10 GB of uint8 images in RAM. Use a **High-RAM** Colab runtime (Runtime -> Change runtime type).

In [ ]:
# ============================================================
# STEP 2: Download datasets
# ============================================================
sys.path.append('/content/sketch_matcher')

from src import download_data

print("Downloading Sketchy Dataset...")
download_data.download_sketchy()

# If automatic download failed, prompt manual upload
sketchy_dir = Path('/content/sketch_matcher/data/raw/sketchy')
if not (sketchy_dir / 'sketch').exists():
    print("\nAutomatic download failed. Manual upload required.")
    print("1. Go to: https://www.kaggle.com/datasets/balraj98/sketchydataset")
    print("2. Click Download (free Kaggle account)")
    print("3. Rename file to 'sketchy_dataset.zip'")
    print("4. Upload using the file picker below:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if 'sketchy' in fn.lower() and fn.endswith('.zip'):
            shutil.move(fn, '/content/sketchy_dataset.zip')
            print(f"Uploaded {fn}. Re-running download...")
            download_data.download_sketchy()

print("\nDownloading QuickDraw subset...")
download_data.download_quickdraw()

print("\nDownloading TU-Berlin (optional, adds sketch variety)...")
download_data.download_tuberlin()

print("\nDownloading ImageNet-Sketch (optional, adds sketch variety)...")
download_data.download_imagenetsketch()

print("\nVerifying...")
download_data.verify_downloads()

---
## Step 3: Preprocess data
Crops, binarizes, resizes to 224x224. Splits by category. Runs ONCE.

In [ ]:
# ============================================================
# STEP 3: Preprocess
# ============================================================
from src import preprocess

print("Preprocessing...")
preprocess.main()
print("Done.")

---
## Step 4: Train Siamese model (3 stages x 2 phases)

| Phase | Stage | What's trainable | Epochs |
|---|---|---|---|
| Teacher (ConvNeXtTiny) | 1 | Dense layers only | 100 |
| Teacher | 2 | Last 12 backbone layers | 60 |
| Teacher | 3 | Full fine-tune | 150 |
| Student (MobileNetV2) | 1 | Dense layers only (distillation) | 100 |
| Student | 2 | Last 12 backbone layers (distillation) | 60 |
| Student | 3 | Full fine-tune (distillation) | 150 |

If `ENABLE_DISTILLATION=False` in config, only the student phase runs.
Runtime on T4: ~2-4 hrs per phase. On H100: much faster.


In [ ]:
# ============================================================
# STEP 4: Train (run overnight ~3.5 hours)
# ============================================================
from src import train

print("Starting training (teacher + student distillation)...")
train.main()
print("\nTraining complete!")

---
## Step 5: Evaluate
Measures Top-1, Top-3, Top-5 accuracy on held-out test categories.

In [ ]:
# ============================================================
# STEP 5: Evaluate
# ============================================================
from src import evaluate

print("Evaluating...")
evaluate.main()
print("Done.")

---
## Step 6: Export to TFLite
Converts embedding model to TFLite int8 (~3.5 MB). Pre-computes photo embeddings for Pi.

In [ ]:
# ============================================================
# STEP 6: Export TFLite
# ============================================================
from src import export_tflite

print("Exporting to TFLite...")
export_tflite.main()
print("\nExport complete!")

---
## Step 7: Download Pi deployment files
Downloads `pi_deploy.zip` to your computer then SCP to Raspberry Pi.

In [ ]:
# ============================================================
# STEP 7: Download Pi files
# ============================================================
pi_dir = Path('/content/sketch_matcher/pi_deploy')
models_dir = Path('/content/sketch_matcher/models')

zip_path = '/content/pi_deploy.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    if pi_dir.exists():
        for f in pi_dir.rglob('*'):
            zf.write(f, f.relative_to(pi_dir.parent))
    if (models_dir / 'sketch_matcher.tflite').exists():
        zf.write(models_dir / 'sketch_matcher.tflite', 'pi_deploy/model_data/sketch_matcher.tflite')
    if (models_dir / 'photo_embeddings.npy').exists():
        zf.write(models_dir / 'photo_embeddings.npy', 'pi_deploy/model_data/photo_embeddings.npy')
    if (models_dir / 'photo_labels.npy').exists():
        zf.write(models_dir / 'photo_labels.npy', 'pi_deploy/model_data/photo_labels.npy')
    if (models_dir / 'labels.json').exists():
        zf.write(models_dir / 'labels.json', 'pi_deploy/model_data/labels.json')

files.download(zip_path)
print("Downloaded!")
print("\nOn your Pi:")
print("  1. Copy zip to Pi")
print("  2. Unzip: unzip pi_deploy.zip")
print("  3. cd pi_deploy && python main.py")

---
## Summary

| Step | Status |
|---|---|
| Data downloaded | Sketchy + TU-Berlin + QuickDraw + ImageNet-Sketch |
| Preprocessed | Crop, binarize, 224x224 |
| Trained | 3 stages x 2 phases (teacher + distilled student) |
| Evaluated | Top-1, Top-3, FAR/FRR/EER |
| Exported | TFLite int8 |
| Downloaded | pi_deploy.zip ready for Pi |

**On your Pi:**
```bash
unzip pi_deploy.zip
cd pi_deploy
pip install opencv-python tflite-runtime picamera2
python main.py
```